# N-BaIoT — Step 5: Feature Selection (All Devices)

This notebook runs Random Forest feature-importance selection separately for every device with existing `train.csv` and `test.csv` files.

For each device it:
1. Samples training rows in chunks, with a maximum number of rows per class.
2. Fits a Random Forest using training data only.
3. Ranks features by impurity-based importance and selects the top `TOP_K`.
4. Writes reduced train and test CSVs using the same selected columns.

**Leakage prevention:** `test.csv` is never used to fit the selector or choose features. It is only reduced after selection.

**Scope:** This is per-device feature selection for within-device experiments. A later cross-device experiment needs a selector fitted only on its training devices.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# CONFIGURATION — run this notebook from the main project folder
DATASET_ROOT = Path.cwd()
PREPROCESSED_ROOT = DATASET_ROOT / "preprocessed"
CHUNK_SIZE = 100_000
MAX_SAMPLES_PER_CLASS = 5_000
TOP_K = 30
N_ESTIMATORS = 200
MAX_DEPTH = 20
MIN_SAMPLES_LEAF = 2
RANDOM_SEED = 42
N_JOBS = -1

print("Preprocessed folder:", PREPROCESSED_ROOT.resolve())
print("Selecting top", TOP_K, "features per device")


## 5.1 Discover completed devices

A device is processed only if both `train.csv` and `test.csv` exist in its `preprocessed/<device>/` folder.


In [ ]:
if not PREPROCESSED_ROOT.is_dir():
    raise FileNotFoundError(f"Cannot find {PREPROCESSED_ROOT}. Check DATASET_ROOT.")

device_dirs = sorted(
    p for p in PREPROCESSED_ROOT.iterdir()
    if p.is_dir() and (p / "train.csv").is_file() and (p / "test.csv").is_file()
)
print("Devices found:", len(device_dirs))
for p in device_dirs:
    print("✓", p.name)
if not device_dirs:
    raise FileNotFoundError("No device train.csv/test.csv pairs found.")


## 5.2 Create a bounded, class-balanced sample from training data

A streaming random-priority reservoir keeps at most `MAX_SAMPLES_PER_CLASS` rows per class without loading the full training CSV into RAM. This makes selector fitting more manageable and limits domination by the largest class.


In [ ]:
def make_training_sample(train_path, feature_cols):
    rng = np.random.default_rng(RANDOM_SEED)
    reservoirs = {}
    seen = {}
    usecols = feature_cols + ["label", "label_id"]

    for chunk_no, chunk in enumerate(
        pd.read_csv(train_path, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False),
        start=1
    ):
        ids = chunk["label_id"].to_numpy(dtype=np.int64)
        X = chunk[feature_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        if not np.isfinite(X).all():
            raise ValueError(
                f"Invalid feature value in {train_path}, chunk {chunk_no}. "
                "Check/re-run preprocessing Step 2."
            )
        labels = chunk["label"].astype(str).to_numpy()

        for class_id in np.unique(ids):
            class_id = int(class_id)
            idx = np.flatnonzero(ids == class_id)
            X_new, y_new = X[idx], labels[idx]
            keys_new = rng.random(len(idx))
            seen[class_id] = seen.get(class_id, 0) + len(idx)

            if class_id in reservoirs:
                keys_old, X_old, y_old = reservoirs[class_id]
                keys = np.concatenate([keys_old, keys_new])
                X_all = np.concatenate([X_old, X_new], axis=0)
                y_all = np.concatenate([y_old, y_new], axis=0)
            else:
                keys, X_all, y_all = keys_new, X_new, y_new

            keep_n = min(MAX_SAMPLES_PER_CLASS, len(keys))
            keep = np.argpartition(keys, keep_n - 1)[:keep_n]
            reservoirs[class_id] = (keys[keep], X_all[keep], y_all[keep])

        if chunk_no % 10 == 0:
            print(f"  Read {chunk_no} chunks")

    if not reservoirs:
        raise ValueError(f"No rows found in {train_path}")

    X_parts, y_parts, report = [], [], {}
    for class_id in sorted(reservoirs):
        _, X_class, _ = reservoirs[class_id]
        X_parts.append(X_class)
        y_parts.append(np.full(len(X_class), class_id, dtype=np.int64))
        report[str(class_id)] = {
            "rows_seen_in_train": int(seen[class_id]),
            "rows_sampled": int(len(X_class))
        }

    X_sample = np.vstack(X_parts)
    y_sample = np.concatenate(y_parts)
    order = rng.permutation(len(y_sample))
    return X_sample[order], y_sample[order], report


def write_reduced_csv(source, destination, selected_features):
    cols = selected_features + ["label", "label_id"]
    if destination.exists():
        destination.unlink()
    first, total = True, 0
    for part in pd.read_csv(source, usecols=cols, chunksize=CHUNK_SIZE, low_memory=False):
        part.to_csv(destination, mode="w" if first else "a", header=first, index=False)
        first = False
        total += len(part)
    if first:
        raise ValueError(f"No rows written from {source}")
    return total


## 5.3 Run Random Forest feature selection for each device

The selector is fitted independently for each device using only a bounded sample from its `train.csv`. Results are saved in `preprocessed/<device>/feature_selection/`.


In [ ]:
results = []

for device_dir in device_dirs:
    name = device_dir.name
    train_path = device_dir / "train.csv"
    test_path = device_dir / "test.csv"
    out_dir = device_dir / "feature_selection"
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 72)
    print("DEVICE:", name)
    print("=" * 72)

    columns = list(pd.read_csv(train_path, nrows=0).columns)
    if not {"label", "label_id"}.issubset(columns):
        raise ValueError(f"{name}: train.csv needs label and label_id columns.")

    features = [c for c in columns if c not in ("label", "label_id")]
    if not features:
        raise ValueError(f"{name}: no feature columns found.")
    if TOP_K > len(features):
        raise ValueError(f"{name}: TOP_K={TOP_K} but only {len(features)} features exist.")

    test_columns = set(pd.read_csv(test_path, nrows=0).columns)
    missing = set(features + ["label", "label_id"]) - test_columns
    if missing:
        raise ValueError(f"{name}: test.csv is missing columns: {sorted(missing)}")

    print("Input feature count:", len(features))
    X_sample, y_sample, sample_report = make_training_sample(train_path, features)
    print("Selector sample:", X_sample.shape, "| classes:", len(np.unique(y_sample)))
    if len(np.unique(y_sample)) < 2:
        raise ValueError(f"{name}: fewer than two classes in selector sample.")

    model = RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    )
    model.fit(X_sample, y_sample)

    ranking = pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False, kind="stable").reset_index(drop=True)
    ranking["rank"] = np.arange(1, len(ranking) + 1)
    ranking["selected"] = ranking["rank"] <= TOP_K
    selected = ranking.loc[ranking["selected"], "feature"].tolist()

    ranking.to_csv(out_dir / "feature_importance.csv", index=False)
    selection = {
        "device": name,
        "method": "Random Forest impurity-based feature importance",
        "selection_scope": "Per-device; fitted on training data only",
        "input_feature_count": len(features),
        "selected_feature_count": len(selected),
        "top_k": TOP_K,
        "selected_features": selected,
        "random_seed": RANDOM_SEED
    }
    with open(out_dir / "selected_features.json", "w", encoding="utf-8") as f:
        json.dump(selection, f, indent=2)

    print("Writing reduced train/test CSVs...")
    n_train = write_reduced_csv(train_path, out_dir / "train_selected.csv", selected)
    n_test = write_reduced_csv(test_path, out_dir / "test_selected.csv", selected)

    report = {
        **selection,
        "selector_model": {
            "type": "RandomForestClassifier",
            "n_estimators": N_ESTIMATORS,
            "max_depth": MAX_DEPTH,
            "min_samples_leaf": MIN_SAMPLES_LEAF,
            "max_features": "sqrt",
            "class_weight": "balanced_subsample"
        },
        "selector_sample_rows": int(len(y_sample)),
        "selector_sample_by_class": sample_report,
        "train_rows_written": int(n_train),
        "test_rows_written": int(n_test),
        "leakage_control": "Test data was not used for fitting or feature selection."
    }
    with open(out_dir / "feature_selection_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    results.append({
        "device": name,
        "status": "completed",
        "original_features": len(features),
        "selected_features": len(selected),
        "selector_sample_rows": len(y_sample),
        "train_rows": n_train,
        "test_rows": n_test,
        "top_feature": selected[0]
    })

    print("Selected top 10:")
    print(ranking.head(10)[["rank", "feature", "importance"]].to_string(index=False))
    print(f"✓ {name}: {len(selected)} of {len(features)} features selected")


## 5.4 Summary and final checks

Each device has its own feature ranking and selected feature set. This is appropriate for separate within-device experiments; it is not a shared cross-device feature set.


In [ ]:
summary = pd.DataFrame(results)
display(summary)

summary_path = PREPROCESSED_ROOT / "all_devices_feature_selection_summary.csv"
summary.to_csv(summary_path, index=False)
print("✓ Step 5 complete.")
print("Summary saved to:", summary_path)


## 5.5 Outputs and what to verify

For each device:

```text
preprocessed/
  Device_Name/
    feature_selection/
      feature_importance.csv
      selected_features.json
      feature_selection_report.json
      train_selected.csv
      test_selected.csv
```

Before Step 6:
- Check whether every expected device appears in the summary.
- Confirm the original feature count (expected 115 in the project plan); investigate any mismatch.
- Review each device's top-ranked features.
- Confirm reduced train/test files contain `TOP_K` features plus `label` and `label_id`.
- Keep the original train/test CSVs unchanged.

**Method limitation:** impurity-based Random Forest importance is a useful baseline, not a definitive measure of feature value. In later evaluation, compare against another selection method or permutation importance if feasible. Fit any scaler or other learned preprocessing on training data only.
